In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df1 = pd.read_pickle(r"S:\Sachuriga\Ephys_Recording\CR_CA1\LFP/LFp.pkl")

In [ ]:
data = []
for idx, row in df1.iterrows():
    animal_id = row['animal_id']
    s = row['session_id'].split("_")
    if s[3]=="A":
        data.append(row)

df = pd.DataFrame(data)

In [ ]:
df

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


#df = pd.read_pickle(r"S:\Sachuriga\Ephys_Recording\CR_CA1\LFP/LFp.pkl")
# Assuming df is the input DataFrame with 'animal_id' and 'lfp_py_norm_run' columns
# Step 1: Get unique animal IDs
unique_animals = np.unique(df['animal_id'])
print(f"Number of unique animals: {len(unique_animals)}")
print(f"Animal IDs: {unique_animals}")


fig = plt.figure(figsize=(7.2, 11), dpi=2400)
plt.rcParams.update({'font.size': 7,'font.family': 'DejaVu Sans'})
gs = gridspec.GridSpec(8, 8, height_ratios=[0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8], width_ratios=[0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8],wspace=3,hspace=3)  # First row taller

plt.rcParams.update({
    'axes.labelpad': -0.1,
    'ytick.major.pad': -0.1,
    'xtick.major.pad': -0.1,
    'ytick.major.size': 2,
    'xtick.major.size': 2
})

ax1 = fig.add_subplot(gs[0:2, 0:4])
ax2 = fig.add_subplot(gs[0:2, 4:8])
ax3 = fig.add_subplot(gs[2:4, 0:4])
ax4 = fig.add_subplot(gs[2:4, 4:8])


ax11 = fig.add_subplot(gs[4:6, 0:2])
ax12 = fig.add_subplot(gs[4:6, 2:4])
ax13 = fig.add_subplot(gs[4:6, 4:6])
ax14 = fig.add_subplot(gs[4:6, 6:8])

ax21 = fig.add_subplot(gs[6:8, 0:2])
ax22 = fig.add_subplot(gs[6:8, 2:4])
ax23 = fig.add_subplot(gs[6:8, 4:6])
ax24 = fig.add_subplot(gs[6:8, 6:8])


axes_event = [ax11, ax12 ,ax13 ,ax14, ax21,ax22 ,ax23 ,ax24]

# Define control and experimental animal IDs
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

# Define a common frequency grid (0 to 100 Hz, assuming 151 points for consistency)
common_frequencies = np.linspace(1, 151, 75)  # Adjust num_points if needed

# Step 2: Collect and average LFP data for each animal with index filtering
animal_lfp_averages = {}
frequency_indices = common_frequencies  # Use common frequencies for plotting

types = ["lfp_py_norm_run","lfp_sr_norm_run","lfp_py_norm_rest","lfp_sr_norm_rest"]
axes = [ax1,ax2,ax3,ax4]

for i,lfp in enumerate(types):
    ax = axes[i]
    for animal_id in unique_animals:
        # Filter DataFrame for the current animal
        animal_data = df[df['animal_id'] == animal_id][lfp]
        
        # Initialize a list to store filtered power values for this animal
        all_power_values = []
        
        # Iterate through each row's power vector
        for power_vector in animal_data:
            if isinstance(power_vector, list) and power_vector:
                # Assume power_vector[0] is a pandas Series with an index
                if isinstance(power_vector[0], pd.Series):
                    power_series = power_vector[0]
                    indices = power_series.index  # Use the Series index directly
                    power_values = power_series.values  # Get the power values
                    
                    # Reindex or interpolate to common_frequencies
                    if not np.array_equal(indices, common_frequencies):
                        # Interpolate to align with common_frequencies
                        interpolated_power = np.interp(
                            common_frequencies,
                            indices,
                            power_values,
                            left=np.nan,
                            right=np.nan
                        )
                    else:
                        interpolated_power = power_values
                    
                    # Filter for indices where 0 <= index <= 100 (already ensured by common_frequencies)
                    if len(interpolated_power) == len(common_frequencies):
                        all_power_values.append(interpolated_power)
                else:
                    print(f"Warning: power_vector[0] for animal {animal_id} is not a pandas Series, skipping.")
        
        # Compute the average power for this animal
        if all_power_values:  # Check if there are any values
            try:
                all_power_values = np.vstack(all_power_values)
                average_power = np.nanmean(all_power_values, axis=0)  # Average across trials, ignoring NaNs
            except ValueError as e:
                print(f"Error stacking arrays for animal {animal_id}: {e}")
                average_power = np.full(len(common_frequencies), np.nan)
        else:
            average_power = np.full(len(common_frequencies), np.nan)  # Handle cases with no data
        
        # Store the result
        animal_lfp_averages[animal_id] = average_power

    # Step 3: Create DataFrame with condition labels
    average_lfp_df = pd.DataFrame({
        "animal_id": animal_lfp_averages.keys(),
        "average_lfp_power": animal_lfp_averages.values()
    })

    # Add condition column
    average_lfp_df['condition'] = average_lfp_df['animal_id'].apply(
        lambda x: 'Control' if x in control_ids else 'Experimental' if x in exp_ids else 'Unknown'
    )

    # Filter out any animals not in control_ids or exp_ids
    average_lfp_df = average_lfp_df[average_lfp_df['condition'] != 'Unknown']

    # Prepare for statistical testing
    control_df = average_lfp_df[average_lfp_df['condition'] == 'Control']
    exp_df = average_lfp_df[average_lfp_df['condition'] == 'Experimental']

    control_powers = np.stack(control_df['average_lfp_power'].values)  # Shape: (num_control_animals, num_freqs)
    exp_powers = np.stack(exp_df['average_lfp_power'].values)  # Shape: (num_exp_animals, num_freqs)

    # Compute p-values using t-test for each frequency bin
    p_values = []
    num_freqs = control_powers.shape[1]
    for i in range(num_freqs):
        ctrl = control_powers[:, i]
        ex = exp_powers[:, i]
        ctrl = ctrl[~np.isnan(ctrl)]
        ex = ex[~np.isnan(ex)]
        if len(ctrl) >= 2 and len(ex) >= 2:
            _, p = scipy.stats.ttest_ind(ctrl, ex)
            p_values.append(p)
        else:
            p_values.append(np.nan)

    p_values = np.array(p_values)

    # FDR correction for multiple comparisons
    valid_mask = ~np.isnan(p_values)
    if np.any(valid_mask):
        reject, q_values, _, _ = multipletests(p_values[valid_mask], method='fdr_bh')
        q_full = np.full_like(p_values, np.nan)
        q_full[valid_mask] = q_values
    else:
        q_full = np.full_like(p_values, np.nan)

    # Identify significant frequencies (q < 0.05)
    significant_mask = q_full < 0.05
    significant_freqs = common_frequencies[significant_mask]

    # Step 4: Prepare data for plotting (convert to long format)
    plot_data = []
    for _, row in average_lfp_df.iterrows():
        animal_id = row['animal_id']
        condition = row['condition']
        power_vector = row['average_lfp_power']
        
        if isinstance(power_vector, np.ndarray) and not np.all(np.isnan(power_vector)):
            for freq_idx, power in zip(frequency_indices, power_vector):
                plot_data.append({
                    'animal_id': animal_id,
                    'condition': condition,
                    'frequency': freq_idx,  # Use common_frequencies for x-axis
                    'power': power
                })

    plot_df = pd.DataFrame(plot_data)

    # Step 5: Create figure with subplot (expand to more subplots as needed, e.g., fig, axs = plt.subplots(1, 2))
    sns.set(style="whitegrid")

    # Plot using lineplot on the specific ax
    sns.lineplot(
        data=plot_df,
        x='frequency',
        y='power',
        hue='condition',
        style='condition',
        palette={'Control': 'blue', 'Experimental': 'red'},
        markers=False,
        dashes=False,
        ax=ax
    )

    # Customize plot
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Normalized Power")
    ax.set_yscale('log')  # Set y-axis to logarithmic scale
    ax.set_ylim(1e-6, 1e-0)  # Set y-axis limits from 10^-6 to 10^0
    ax.set_xlim(0, 150)  # Adjust x-limit to 0-100 Hz as per original title
    ax.grid(False)
    ax.set_aspect('auto')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.legend().set_visible(False)
    # Add green dots at the top for significant differences
    if len(significant_freqs) > 0:
        y_pos = 5e-1  # Position near the top of the y-axis (adjust if needed based on data)
        ax.scatter(significant_freqs, [y_pos] * len(significant_freqs), color='green', s=20, marker='*')

    # Step 6: Display the DataFrame for reference
    print("\nAverage LFP DataFrame with Conditions:")
    print(average_lfp_df[['animal_id', 'condition']])

# Assuming df is the input DataFrame with 'animal_id' and 'lfp_py_norm_run' columns
# Step 1: Get unique animal IDs
unique_animals = np.unique(df['animal_id'])
print(f"Number of unique animals: {len(unique_animals)}")
print(f"Animal IDs: {unique_animals}")



import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import shapiro, ttest_ind, mannwhitneyu

# Assuming df is your DataFrame
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']


variables = ['slow_event_rate_py', 'slow_theta_gamma_coupling_py','fast_event_rate_py', 'fast_theta_gamma_coupling_py', 
            "lfp_sr_norm_run_fast_gamma_sum", "lfp_sr_norm_run_slow_gamma_sum","lfp_py_norm_run_fast_gamma_sum", "lfp_py_norm_run_slow_gamma_sum"]

# variables = ['slow_event_rate_py', 'slow_theta_gamma_coupling_py','fast_event_rate_py', 'fast_theta_gamma_coupling_py', 
#             'slow_event_rate_sr', 'slow_theta_gamma_coupling_sr','fast_event_rate_sr', 'fast_theta_gamma_coupling_sr']


titles = ['Episods/s', 'Vector length','Episods/s', 'Vector length', 
            "Theta power durnig run", "Theta power durnig rest","Gamma power during run", "Gamma power during rest"]

# Initialize an empty list to store results
data = []
unpacked_data=[]

for idx, row in df.iterrows():
    animal_id = row['animal_id']
    condition = "Control" if animal_id in control_ids else "Exp" if animal_id in exp_ids else None
    if condition:
        row_data = {"condition": condition, "row_id": idx,'animal_id':animal_id}
        for var in variables:
            value = row[var]  # Assume scalar for simplicity
            row_data[var] = value
        data.append(row_data)

data_df = pd.DataFrame(data)

for _, row in data_df.iterrows():
    condition = row['condition']
    row_id = row['row_id']
    animal_id = row['animal_id']
    # Get the lists for each variable

    for index,var in enumerate(variables):
        lists_per_var = {var: row[var] for var in variables}
        # Determine the length of the lists (assuming all lists in a row have the same length)
        try:
            list_length = len(lists_per_var[variables[index]]) if lists_per_var[variables[index]] else 0
            # Create a row for each index in the lists
            for i in range(list_length):
                new_row = {
                    'condition': condition,
                    'row_id': row_id,
                    'list_index': i,  # To track the position in the list
                    'animal_id':animal_id
                }
            new_row[var] = lists_per_var[var][i] if i < len(lists_per_var[var]) else None
            unpacked_data.append(new_row)
        except Exception as e:

            new_row[var] = lists_per_var[var]
            unpacked_data.append(new_row)

    

# Create a new DataFrame from the unpacked data
unpacked_df = pd.DataFrame(unpacked_data)
control_color = 'blue'
exp_color = 'red'
# Create subplots

# cols = [
#     'slow_event_rate_py', 'slow_theta_gamma_coupling_py',
#     'fast_event_rate_py', 'fast_theta_gamma_coupling_py',
#     'slow_event_rate_sr', 'slow_theta_gamma_coupling_sr',
#     'fast_event_rate_sr', 'fast_theta_gamma_coupling_sr'
# ]
# dff=unpacked_df
# # (optional) ensure these columns are numeric
# dff[cols] = dff[cols].apply(pd.to_numeric, errors='coerce')

# # group by animal id and take the mean
# df_avg = (
#     dff.groupby('animal_id', as_index=False)[cols]
#       .mean()
# )
# control_ids = ['65165', '65091', '63383', '66539', '65622']
# exp_ids     = ['65588', '63385', '66538', '66537', '66922']

# # ensure ids are strings
# df_avg['animal_id'] = df_avg['animal_id'].astype(str)

# # option A: with a mapping (cleanest)
# mapping = {**{i: 'Control' for i in control_ids},
#            **{i: 'Exp'     for i in exp_ids}}
# df_avg['condition'] = df_avg['animal_id'].map(mapping).fillna('Unknown')

# # If you already computed df_avg (from your previous step), add the same column:
# df_avg['condition'] = df_avg['animal_id'].map(mapping).fillna('Unknown')

# unpacked_df = df_avg
for i, var in enumerate(variables):
    # Create subplot
    ax = axes_event[i]
    # # Boxplot with specified colors
    # sns.boxplot(x='condition', y=var, data=unpacked_df, 
    #             palette={'Control': 'blue', 'Exp': 'cyan'})
    
    sns.violinplot(
            data=unpacked_df, x='condition', y=var, 
            ax=ax,
            inner = None,
            palette={"Control": control_color, "Exp": exp_color}, width=0.8, cut=0, linewidth=0
        )
    # Add individual points with matching colors
    sns.boxplot(
        data=unpacked_df, 
        x='condition', y=var, 
        palette={"Control": "black", "Exp": "black"},
        width=0.3, 
        fill=False,  # No fill, only outlines
        showfliers=False,  # Hide outliers
        showmeans=False,  # Remove mean marker, assuming midline is the median
        linewidth=1,  # Makes the lines narrower (thinner)
        ax=ax  # Add this
    )
    #ax.set_ylabel(titles[i])
    ax.set_xlabel('')
    ax.yaxis.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    ax.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation = -45)
    y_max = ax.get_ylim()[1]

    ax.set_ylabel(titles[i])
    # Normality test
    control_data = unpacked_df[unpacked_df['condition'] == 'Control'][var].dropna()
    exp_data = unpacked_df[unpacked_df['condition'] == 'Exp'][var].dropna()
    
    # Shapiro test for normality
    stat_c, p_c = shapiro(control_data)
    stat_e, p_e = shapiro(exp_data)
    
    # Choose statistical test based on normality (p < 0.05 indicates non-normal)
    if p_c > 0.05 and p_e > 0.05:
        # Both normal: use t-test
        stat, p_val = ttest_ind(control_data, exp_data)
        test_name = 't-test'
    else:
        # At least one non-normal: use Mann-Whitney U
        stat, p_val = mannwhitneyu(control_data, exp_data)
        test_name = 'Mann-Whitney U'
    
    # Add title with statistical results
    y_max = ax.get_ylim()[1]
    bar_height = y_max * 0.1  # Adjust this value to position the bar above the plot
    x_positions = [0, 1]  # Adjusted positions for 'Control' and 'Experimental' groups
    p_val =  p_val
    if (p_val < 0.05) & (p_val > 0.01):
        ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                            color='black', lw=1.5)
        ax.text(0.5, y_max + bar_height * 1.1, f'*', ha='center', va='bottom')
    elif (p_val < 0.01) & (p_val > 0.001):
        ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                            color='black', lw=1.5)
        ax.text(0.5, y_max + bar_height * 1.1, f'**', ha='center', va='bottom')
    elif p_val < 0.001:
        ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                            color='black', lw=1.5)
        ax.text(0.5, y_max + bar_height * 1.1, f'***', ha='center', va='bottom')


# Print detailed statistical results
print("\nStatistical Analysis Results:")
print("-" * 50)
for var in variables:
    control_data = unpacked_df[unpacked_df['condition'] == 'Control'][var].dropna()
    exp_data = unpacked_df[unpacked_df['condition'] == 'Exp'][var].dropna()
    
    stat_c, p_c = shapiro(control_data)
    stat_e, p_e = shapiro(exp_data)
    
    if p_c > 0.05 and p_e > 0.05:
        stat, p_val = ttest_ind(control_data, exp_data)
        test_name = 't-test'
    else:
        stat, p_val = mannwhitneyu(control_data, exp_data)
        test_name = 'Mann-Whitney U'
    
    print(f"\n{var}:")
    print(f"Control normality (Shapiro): p={p_c:.4f}")
    print(f"Exp normality (Shapiro): p={p_e:.4f}")
    print(f"{test_name}: statistic={stat:.4f}, p-value={p_val:.4f}")

#     # Adjust layout and display
# plt.tight_layout()
# plt.savefig(r'Q:/sachuriga/CR_CA1_paper/Figures/suppfig10.png', transparent=True, dpi=1200, bbox_inches='tight')
# plt.show()

In [ ]:
unpacked_df

In [ ]:
variables['slow_event_rate_py']

In [ ]:
list_length

In [ ]:
len(lists_per_var[var])

In [ ]:
# --- Add theta/gamma band-power sums to the ORIGINAL df ---
common_frequencies = np.linspace(1, 151, 75) 
# Bands (Hz)
THETA_BAND = (4.0, 12.0)
SLOW_GAMMA_BAND = (20.0, 40)
GAMMA_BAND = (40.0, 91.0)

# LFP columns you already use
lfp_cols = ["lfp_py_norm_run","lfp_sr_norm_run","lfp_py_norm_rest","lfp_sr_norm_rest"]

def sum_band_power_from_cell(cell, band, common_frequencies):
    """
    Extracts the power spectrum from a df cell (expected: [pd.Series]) and returns
    the SUM of power within the specified band after aligning to common_frequencies.
    Returns np.nan when data are missing/ill-formed.
    """
    #try:
    if isinstance(cell, list) and len(cell) > 0 and isinstance(cell[0], pd.Series):
        s = cell[0]
        freqs_src = s.index.values.astype(float)
        power_src = s.values.astype(float)

        # Align to target frequency grid if needed
        if not np.array_equal(freqs_src, common_frequencies):
            power_interp = np.interp(
                common_frequencies, freqs_src, power_src,
                left=np.nan, right=np.nan
            )
            freqs = common_frequencies
            power = power_interp
        else:
            freqs = freqs_src
            power = power_src

        # Band mask (inclusive)
        mask = (freqs >= band[0]) & (freqs <= band[1])
        if not np.any(mask):
            return np.nan

        # Sum of power in band (as requested). If you prefer area, use np.trapz instead.
        band_values = power[mask]
        return np.nansum(band_values)
    # except Exception:
    #     pass
    # return np.nan

# Compute and attach columns
for col in lfp_cols:
    theta_col = f"{col}_theta_sum"
    gamma_col = f"{col}_fast_gamma_sum"
    slow_gamma_col = f"{col}_slow_gamma_sum"
    df[theta_col] = df[col].apply(lambda cell: sum_band_power_from_cell(cell, THETA_BAND, common_frequencies))
    df[gamma_col] = df[col].apply(lambda cell: sum_band_power_from_cell(cell, GAMMA_BAND, common_frequencies))
    df[slow_gamma_col] = df[col].apply(lambda cell: sum_band_power_from_cell(cell, SLOW_GAMMA_BAND, common_frequencies))

# (Optional) quick peek
print(df[[c for col in lfp_cols for c in (f"{col}_theta_sum", f"{col}_fast_gamma_sum")]].head())


In [ ]:
df

In [ ]:
cols = [
    'slow_event_rate_py', 'slow_theta_gamma_coupling_py',
    'fast_event_rate_py', 'fast_theta_gamma_coupling_py',
    'slow_event_rate_sr', 'slow_theta_gamma_coupling_sr',
    'fast_event_rate_sr', 'fast_theta_gamma_coupling_sr'
]

# (optional) ensure these columns are numeric
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

# group by animal id and take the mean
df_avg = (
    df.groupby('animal_id', as_index=False)[cols]
      .mean()
)
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids     = ['65588', '63385', '66538', '66537', '66922']

# ensure ids are strings
df_avg['animal_id'] = df_avg['animal_id'].astype(str)

# option A: with a mapping (cleanest)
mapping = {**{i: 'Control' for i in control_ids},
           **{i: 'Exp'     for i in exp_ids}}
df_avg['condition'] = df_avg['animal_id'].map(mapping).fillna('Unknown')

# If you already computed df_avg (from your previous step), add the same column:
df_avg['condition'] = df_avg['animal_id'].map(mapping).fillna('Unknown')

In [ ]:
df_avg

In [ ]:
import numpy as np
from scipy.signal import butter, sosfiltfilt, argrelextrema, hilbert
import matplotlib.pyplot as plt

def bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    sos = butter(order, [low, high], analog=False, btype='band', output='sos')
    y = sosfiltfilt(sos, data)
    return y

def generate_normalized_spectrogram(signal, fs=1250, freq_step=2, num_phase_bins=12):
    # Bandpass filter for theta (6-10 Hz)
    theta_filt = bandpass_filter(signal, 6, 12, fs)
    
    # Find local minima (troughs)
    minima_idx = argrelextrema(theta_filt, np.less)[0]
    
    # Select theta cycles based on duration criteria
    cycle_pairs = []
    for i in range(len(minima_idx) - 1):
        start = minima_idx[i]
        end = minima_idx[i + 1]
        dur_ms = (end - start) / fs * 1000
        if 100 <= dur_ms <= 150:
            cycle_pairs.append((start, end))
    
    if not cycle_pairs:
        raise ValueError("No valid theta cycles found.")
    
    # Compute Hilbert transform for phase
    analytic = hilbert(theta_filt)
    phase = np.angle(analytic)
    
    # Adjust phase so that average trough phase is 0
    trough_phases = phase[minima_idx] % (2 * np.pi)
    mean_trough_phase = np.mean(trough_phases)
    adjusted_phase = (phase - mean_trough_phase) % (2 * np.pi)
    
    # Frequencies from 2 to 140 Hz in 2 Hz steps
    freqs = np.arange(12, 121, freq_step)
    
    # Phase bins (0 to 360 degrees, num_bins bins)
    bin_edges = np.linspace(0, 2 * np.pi, num_phase_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Accumulators for average power
    power_sum = np.zeros((len(freqs), num_phase_bins))
    count = np.zeros((len(freqs), num_phase_bins))
    
    dt = 1 / fs
    
    for start, end in cycle_pairs:
        segment = signal[start:end]
        cycle_phases = adjusted_phase[start:end]
        
        for f_idx, f in enumerate(freqs):
            sigma_f = f / 7.0
            sigma_t = 1 / (2 * np.pi * sigma_f)
            
            # Wavelet support: +/- 4 sigma_t
            t_wave = np.arange(-4 * sigma_t, 4 * sigma_t + dt, dt)
            A = 1 / np.sqrt(sigma_t * np.sqrt(np.pi))
            wavelet = A * np.exp(-t_wave**2 / (2 * sigma_t**2)) * np.exp(2j * np.pi * f * t_wave)
            
            # Pad the segment asymmetrically if needed to make conv 'valid' output len(segment)
            pad_left = (len(wavelet) - 1) // 2
            pad_right = len(wavelet) - 1 - pad_left
            padded_segment = np.pad(segment, (pad_left, pad_right), mode='reflect')
            
            # Convolve with 'valid' to get output same length as segment
            conv = np.convolve(padded_segment, wavelet, mode='valid')
            power = np.abs(conv)**2
            
            # Bin powers by phase
            bin_idx = np.digitize(cycle_phases, bin_edges) - 1
            for b in range(num_phase_bins):
                mask = (bin_idx == b)
                if np.any(mask):
                    power_sum[f_idx, b] += np.sum(power[mask])
                    count[f_idx, b] += np.sum(mask)
    
    # Compute average power
    avg_power = np.divide(power_sum, count, where=count > 0)
    avg_power[count == 0] = np.nan  # Handle empty bins
    
    # Normalize for each frequency: divide by mean across phases
    norm_spectrogram = np.zeros_like(avg_power)
    for f_idx in range(len(freqs)):
        mean_p = np.nanmean(avg_power[f_idx, :])
        if mean_p > 0:
            norm_spectrogram[f_idx, :] = avg_power[f_idx, :] / mean_p
        else:
            norm_spectrogram[f_idx, :] = np.nan
    
    return freqs, bin_centers / np.pi * 180, norm_spectrogram  # phases in degrees

# Example usage (replace with your actual signal)
# signal = your_lfp_data_here  # np.array of LFP signal

# For demonstration, generate a sample signal (replace with real data)
fs = 1250
signal = eeg
freqs, phases, norm_spectrogram = generate_normalized_spectrogram(signal, fs)
# Plot the figure
fig, ax = plt.subplots(figsize=(8, 6))
pcm = ax.pcolormesh(phases, freqs, norm_spectrogram, shading='gouraud', cmap='jet')
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label('Normalized Power')
ax.set_xlabel('Theta Phase (degrees)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('Normalized Power Spectrogram Averaged Across Theta Cycles')
plt.show()

In [ ]:
from nwb4fp.analyses.data import pos2speed,speed_filtered_spikes,load_speed_fromNWB,load_units_fromNWB,find_run_indices
import pynapple as nap
i=3
smoothed_speed = df['smoothed_speed'][i]
time_stemp = df['time_stemp'][i]
starts,stops = find_run_indices(smoothed_speed, threshold=0.05)
run_ep = nap.IntervalSet(start=time_stemp[starts], end=time_stemp[stops], time_units='s')

eeg = df['lfp_py'][3][0].restrict(run_ep).values


In [ ]:
# Plot the figure
fig, ax = plt.subplots(figsize=(8, 6))
signal = df['lfp_py'][85][0].values
freqs, phases, norm_spectrogram = generate_normalized_spectrogram(signal, fs)
pcm = ax.pcolormesh(phases, freqs, norm_spectrogram, shading='gouraud', cmap='jet')
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label('Normalized Power')
ax.set_xlabel('Theta Phase (degrees)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('Normalized Power Spectrogram Averaged Across Theta Cycles')
plt.show()

In [ ]:
eeg = df['lfp_py'][3][0].values

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, hilbert

def _bandpass_sos(low, high, fs, order=4):
    nyq = fs / 2.0
    if low <= 0:
        low = 0.5  # avoid DC; tweak if you truly need ultra-low bands
    sos = butter(order, [low/nyq, high/nyq], btype='bandpass', output='sos')
    return sos

def _analytic_band(x, fs, f_lo, f_hi):
    """Bandpass then analytic signal (Hilbert)."""
    sos = _bandpass_sos(f_lo, f_hi, fs)
    xf = sosfiltfilt(sos, x)
    z = hilbert(xf)
    return z  # complex analytic signal

def _centered_bins(f_min, f_max, step, width):
    """Return list of (f_lo, f_hi) around sliding centers with constant width."""
    centers = np.arange(f_min, f_max + 1e-9, step)
    bands = [(max(0.1, c - width/2), c + width/2) for c in centers]
    return centers, bands

def cross_frequency_comodulogram(
    eeg, fs, channel=0,
    phase_range=(2, 20), phase_step=1, phase_bw=2,      # Hz
    amp_range=(30, 150), amp_step=2, amp_bw=10,         # Hz
    method="PAC",                                       # "PAC", "AAC", or "PPLV"
    nm_ratio=(1, 1),                                    # for PPLV: n:m (e.g., (1,2))
    detrend=True,
    trim=1.0,                                           # seconds trimmed at start/end to avoid filter edge
    plot=True,
    cmap="viridis",
    return_parts=False,                                  # also return per-band signals used
):
    """
    Compute and (optionally) plot a cross-frequency 'coherence' style comodulogram.

    Parameters
    ----------
    eeg : array_like, shape (n_channels, n_samples) or (n_samples,)
        EEG data.
    fs : float
        Sampling rate (Hz).
    channel : int or tuple
        If method=="AAC" and you want cross-channel AAC, pass (ch_amp, ch_phase) or (ch1, ch2).
        Otherwise a single channel index. If `eeg` is 1D, this is ignored.
    phase_range, amp_range : (f_min, f_max)
        Frequency search ranges (Hz) for phase (rows) and amplitude (cols).
    phase_step, amp_step : float
        Step size (Hz) of the center frequencies.
    phase_bw, amp_bw : float
        Bandwidths (Hz) around each center.
    method : {"PAC", "AAC", "PPLV"}
        - PAC  : |mean( zscore(A_high) * exp(1j*phi_low) )|
        - AAC  : Pearson r between amplitude envelopes across bands (same channel by default).
        - PPLV : |mean( exp( i*( n*phi_low - m*phi_high ) ) )| with nm_ratio=(n,m).
    nm_ratio : (int,int)
        n:m for phase-phase locking (PPLV).
    detrend : bool
        Remove mean from signals before filtering.
    trim : float
        Seconds to drop from start and end to reduce filter edge effects.
    plot : bool
        Show a heatmap.
    cmap : str
        Matplotlib colormap.
    return_parts : bool
        If True, also returns dict of intermediate signals for debugging.

    Returns
    -------
    freqs_phase : ndarray, shape (nP,)
    freqs_amp   : ndarray, shape (nA,)
    C           : ndarray, shape (nP, nA)
    parts (optional) : dict
    """
    x = np.asarray(eeg)
    if x.ndim == 1:
        X = x[None, :]
    else:
        X = x

    # choose channels
    if isinstance(channel, tuple) and len(channel) == 2:
        ch_phase = channel[0]
        ch_amp   = channel[1]
    else:
        ch_phase = ch_amp = channel

    sig_phase = X[ch_phase].astype(float).copy()
    sig_amp   = X[ch_amp].astype(float).copy()

    if detrend:
        sig_phase -= sig_phase.mean()
        sig_amp   -= sig_amp.mean()

    n = sig_phase.size
    drop = int(trim * fs)
    keep = slice(drop, n - drop if (n - drop) > drop else n)

    # frequency grids
    fP, bandsP = _centered_bins(phase_range[0], phase_range[1], phase_step, phase_bw)
    fA, bandsA = _centered_bins(amp_range[0], amp_range[1], amp_step, amp_bw)

    # Precompute analytic signals for efficiency
    phase_an = []
    for (flo, fhi) in bandsP:
        z = _analytic_band(sig_phase, fs, flo, fhi)[keep]
        phase_an.append(z)

    amp_an = []
    for (flo, fhi) in bandsA:
        z = _analytic_band(sig_amp, fs, flo, fhi)[keep]
        amp_an.append(z)

    C = np.zeros((len(fP), len(fA)))

    if method.upper() == "PAC":
        # PAC: magnitude of mean( z(A) * e^{i*phi} )
        for ip, zP in enumerate(phase_an):
            phi = np.angle(zP)
            eiphi = np.exp(1j * phi)
            for ia, zA in enumerate(amp_an):
                A = np.abs(zA)
                A = (A - A.mean()) / (A.std(ddof=1) + 1e-12)  # z-score to mitigate amplitude bias
                C[ip, ia] = np.abs(np.mean(A * eiphi))

    elif method.upper() == "AAC":
        # AAC: correlation between amplitude envelopes
        for ip, zP in enumerate(phase_an):
            AP = np.abs(zP)
            for ia, zA in enumerate(amp_an):
                AA = np.abs(zA)
                # Same length by construction
                r = np.corrcoef(AP, AA)[0, 1]
                C[ip, ia] = r

        # make all positive by absolute value (optional)
        # C = np.abs(C)

    elif method.upper() == "PPLV":
        n_h, m_h = nm_ratio
        for ip, zP in enumerate(phase_an):
            phiP = np.angle(zP)
            for ia, zA in enumerate(amp_an):
                phiA = np.angle(zA)
                plv = np.abs(np.mean(np.exp(1j * (n_h * phiP - m_h * phiA))))
                C[ip, ia] = plv
    else:
        raise ValueError("method must be one of {'PAC','AAC','PPLV'}")

    if plot:
        fig, ax = plt.subplots(figsize=(7, 5), dpi=140)
        im = ax.imshow(
            C,
            origin="lower",
            aspect="auto",
            extent=[fA[0], fA[-1], fP[0], fP[-1]],
            cmap=cmap,
        )
        ax.set_xlabel("Amplitude frequency (Hz)" if method.upper() != "PPLV" else "Higher phase frequency (Hz)")
        ax.set_ylabel("Phase frequency (Hz)")
        ax.set_title(f"Cross-frequency {method.upper()} comodulogram")
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label({
            "PAC": "PAC (|mean(z(A) e^{iφ})|)",
            "AAC": "Envelope correlation (r)",
            "PPLV": "n:m PLV"
        }[method.upper()])
        plt.tight_layout()

    parts = None
    if return_parts:
        parts = dict(
            phase_analytic=phase_an,
            amp_analytic=amp_an,
            keep_idx=keep,
            f_phase=fP,
            f_amp=fA,
        )
        return fP, fA, C, parts
    return fP, fA, C


# -------------------------
# Example usage
# -------------------------

# Fake 10 s of data with 1 kHz sampling
fs = 1250.0
t = np.arange(0, len(eeg)/fs, 1/fs)


# Compute and show PAC comodulogram for channel 0
cross_frequency_comodulogram(
    eeg, fs, channel=0,
    phase_range=(0, 100), phase_step=1, phase_bw=2,
    amp_range=(0, 100), amp_step=2, amp_bw=10,
    method="PAC", plot=True
)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_cfc_like_paper(C1, f_phase, f_power, C2=None,
                        vmax=0.2, hline=40, panel_titles=("",""),
                        cmap="jet", figsize=(8,3.4)):
    """
    Make a CFC comodulogram styled like the reference figure.

    Parameters
    ----------
    C1 : (nP, nA) ndarray
        Comodulogram values (rows: phase freqs, cols: power freqs).
    f_phase : (nP,) ndarray
        Phase frequency grid (Hz) for x-axis.
    f_power : (nA,) ndarray
        Power/envelope frequency grid (Hz) for y-axis.
    C2 : (nP, nA) ndarray or None
        Optional second panel (e.g., control/condition B).
    vmax : float
        Upper limit for colorbar; vmin is 0.
    hline : float
        Horizontal dashed line (Hz), e.g., 40 Hz.
    panel_titles : tuple[str,str]
        Titles for left and right panels.
    cmap : str
        Matplotlib colormap (paper-like look: "jet" or "turbo").
    figsize : tuple
        Figure size in inches.

    Returns
    -------
    fig, axes
    """
    panels = 2 if C2 is not None else 1
    fig, axes = plt.subplots(1, panels, figsize=figsize, dpi=160, constrained_layout=True)

    if panels == 1:
        axes = [axes]

    def _draw(ax, C, title):
        im = ax.imshow(
            C.T,                   # transpose so y-axis is power
            origin="lower",
            aspect="auto",
            extent=[f_phase[0], f_phase[-1], f_power[0], f_power[-1]],
            vmin=0, vmax=vmax, cmap=cmap
        )
        ax.axhline(hline, ls="--", lw=1.0, color="w", alpha=0.8)
        ax.set_xlim(0, max(100, f_phase[-1]))
        ax.set_ylim(0, max(100, f_power[-1]))
        ax.set_xlabel("Frequency_phase (Hz)")
        ax.set_ylabel("Frequency_power (Hz)")
        if title:
            ax.set_title(title, pad=2, fontsize=9)
        return im

    im = _draw(axes[0], C1, panel_titles[0])
    if panels == 2:
        _ = _draw(axes[1], C2, panel_titles[1])

    cbar = fig.colorbar(im, ax=axes, fraction=0.046, pad=0.04)
    cbar.set_label("", rotation=0, labelpad=0)
    return fig, axes

# -------------------------
# Minimal example
# -------------------------
if __name__ == "__main__":
    # Create fake grids (0–100 Hz)
    fP = np.arange(1, 101)      # phase
    fA = np.arange(1, 101)      # power/envelope

    # Synthesize a left panel with strong 4–12 Hz phase modulating 60–90 Hz power
    C_left = np.zeros((len(fP), len(fA)))
    for i, fp in enumerate(fP):
        for j, fa in enumerate(fA):
            bump1 = np.exp(-0.5*((fp-8)/3)**2) * np.exp(-0.5*((fa-80)/10)**2)
            bump2 = 0.6*np.exp(-0.5*((fp-12)/5)**2) * np.exp(-0.5*((fa-55)/8)**2)
            C_left[i, j] = bump1 + bump2
    C_left /= C_left.max() * 5.0  # scale so peak ~0.2

    # Right panel as a “control” with only weak low-freq coupling
    C_right = 0.4*np.exp(-0.5*((fP[:,None]-6)/2.5)**2) * np.exp(-0.5*((fA[None,:]-18)/5)**2)
    C_right /= C_right.max() * 5.0

    plot_cfc_like_paper(C_left, fP, fA, C2=C_right,
                        vmax=0.2, hline=40,
                        panel_titles=("Condition A", "Condition B"))
    plt.show()


In [ ]:
"""
Phase–Amplitude Coupling (PAC) comodulogram with Tort Modulation Index (MI)

This module provides:
  1) A robust MI implementation (Tort et al., 2010 style) for limited-time datasets
  2) Time-stepped / event-epoch PAC computation
  3) Publication-ready plotting helpers for comodulograms

Primary entry point: `compute_pac_comodulogram` + `plot_comodulogram`.

Author: ChatGPT
"""
from __future__ import annotations
from typing import Iterable, List, Optional, Sequence, Tuple
import numpy as np
from scipy.signal import butter, filtfilt, hilbert
import matplotlib.pyplot as plt

ArrayLike = np.ndarray

# ------------------------------
# Filtering utilities
# ------------------------------
def _butter_bandpass(low: float, high: float, fs: float, order: int = 4) -> Tuple[np.ndarray, np.ndarray]:
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq
    if low_n <= 0 or high_n >= 1 or low_n >= high_n:
        raise ValueError(f"Invalid band [{low}, {high}] Hz for fs={fs} Hz.")
    b, a = butter(order, [low_n, high_n], btype="bandpass")
    return b, a


def bandpass_filter(x: ArrayLike, fs: float, band: Tuple[float, float], order: int = 4) -> ArrayLike:
    """Zero-phase IIR bandpass using filtfilt.

    Parameters
    ----------
    x : array, shape (n_samples,) or (n_channels, n_samples)
    fs : float
        Sampling rate in Hz.
    band : (low, high)
        Frequency band in Hz.
    order : int
        Butterworth order.
    """
    x = np.asarray(x)
    b, a = _butter_bandpass(band[0], band[1], fs, order)
    if x.ndim == 1:
        return filtfilt(b, a, x)
    elif x.ndim == 2:
        return np.vstack([filtfilt(b, a, xi) for xi in x])
    else:
        raise ValueError("x must be 1D or 2D")


# ------------------------------
# Modulation Index (Tort et al.)
# ------------------------------

def modulation_index_tort(phase_series: ArrayLike, amp_envelope: ArrayLike, n_bins: int = 18, eps: float = 1e-10) -> float:
    """Compute Tort modulation index between a phase time series and an amplitude envelope.

    Parameters
    ----------
    phase_series : array
        Instantaneous phase in radians (e.g., angle(hilbert(theta_band))).
    amp_envelope : array
        Instantaneous amplitude envelope (e.g., abs(hilbert(gamma_band))).
    n_bins : int
        Number of phase bins spanning [-pi, pi).
    eps : float
        Small value to avoid log(0).

    Returns
    -------
    MI : float
        Normalized KL divergence between the phase-binned amplitude distribution and uniform.
    """
    phase = np.asarray(phase_series)
    amp = np.asarray(amp_envelope)
    mask = np.isfinite(phase) & np.isfinite(amp)
    phase = phase[mask]
    amp = amp[mask]
    if phase.size < n_bins * 5:
        return np.nan  # too few samples for a stable estimate

    # Bin by phase
    edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    # Map phase to bin indices [0, n_bins-1]
    idx = np.digitize(phase, edges) - 1
    idx[idx == n_bins] = n_bins - 1

    # Mean amplitude per phase bin
    mean_amp = np.zeros(n_bins)
    counts = np.zeros(n_bins, dtype=int)
    for k in range(n_bins):
        sel = idx == k
        counts[k] = sel.sum()
        if counts[k] > 0:
            mean_amp[k] = amp[sel].mean()
        else:
            mean_amp[k] = 0.0

    if counts.sum() == 0:
        return np.nan

    P = mean_amp / (mean_amp.sum() + eps)
    U = np.full(n_bins, 1.0 / n_bins)

    # KL divergence and normalization
    kl = np.sum(P * (np.log(P + eps) - np.log(U + eps)))
    mi = kl / np.log(n_bins)
    return float(mi)


# ------------------------------
# Epoch helpers
# ------------------------------

def make_sliding_epochs(n_samples: int, fs: float, window_s: float, step_s: Optional[float] = None) -> List[Tuple[int, int]]:
    """Create [start, stop) sample windows covering the signal.

    If step_s is None, defaults to window_s/2.
    """
    if step_s is None:
        step_s = window_s / 2
    w = int(round(window_s * fs))
    s = int(round(step_s * fs))
    if w <= 1:
        raise ValueError("window too small")
    starts = np.arange(0, n_samples - w + 1, s)
    return [(int(a), int(a + w)) for a in starts]


# ------------------------------
# Core PAC computation
# ------------------------------

def compute_pac_comodulogram(
    lfp: ArrayLike,
    fs: float,
    phase_bands: Sequence[Tuple[float, float]],
    amp_bands: Sequence[Tuple[float, float]],
    epochs: Optional[Sequence[Tuple[int, int]]] = None,
    window_s: Optional[float] = None,
    step_s: Optional[float] = None,
    n_bins: int = 18,
    filter_order: int = 4,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Optional[np.ndarray]]:
    """Compute PAC MI comodulogram over frequency pairs and (optionally) time epochs.

    You can provide explicit `epochs` as a list of (start, stop) sample indices.
    Alternatively, set `window_s` (and optional `step_s`) to tile the signal.

    Parameters
    ----------
    lfp : array, shape (n_samples,)
        LFP / field potential time series.
    fs : float
        Sampling rate in Hz.
    phase_bands : list of (f_lo, f_hi)
        Bands that provide phase (typically low frequencies, e.g., theta 4–12 Hz).
    amp_bands : list of (f_lo, f_hi)
        Bands whose amplitude is modulated (typically higher, e.g., gamma 30–200 Hz).
    epochs : list of (start, stop) in samples, optional
        Event-aligned epochs. If None and window_s given, sliding windows are used.
    window_s, step_s : float, optional
        Sliding window parameters in seconds.
    n_bins : int
        Number of phase bins for MI.
    filter_order : int
        Butterworth order for bandpass.

    Returns
    -------
    mi : array, shape (n_phase, n_amp, n_epochs)
    phase_centers : array, shape (n_phase,)
    amp_centers : array, shape (n_amp,)
    epoch_times_s : array, shape (n_epochs,), optional
        Epoch center times in seconds (None if `epochs` provided explicitly without timing context).
    """
    x = np.asarray(lfp).astype(float)
    if x.ndim != 1:
        raise ValueError("lfp must be 1D")

    # Epochs
    if epochs is None:
        if window_s is None:
            raise ValueError("Provide `epochs` or `window_s`.")
        epochs = make_sliding_epochs(len(x), fs, window_s, step_s)
        epoch_times_s = np.array([(a + b) / 2 / fs for a, b in epochs])
    else:
        epoch_times_s = np.array([(a + b) / 2 / fs for a, b in epochs])

    nP = len(phase_bands)
    nA = len(amp_bands)
    nE = len(epochs)

    # Pre-filter once per band to avoid re-filtering per epoch
    phase_filt = []  # list of instantaneous phase arrays
    for (flo, fhi) in phase_bands:
        xf = bandpass_filter(x, fs, (flo, fhi), order=filter_order)
        phase_filt.append(np.angle(hilbert(xf)))
    phase_filt = np.stack(phase_filt, axis=0)  # (nP, n_samples)

    amp_filt = []  # list of amplitude envelopes per amp band
    for (flo, fhi) in amp_bands:
        xf = bandpass_filter(x, fs, (flo, fhi), order=filter_order)
        amp_filt.append(np.abs(hilbert(xf)))
    amp_filt = np.stack(amp_filt, axis=0)  # (nA, n_samples)

    # Centers for axes labels
    phase_centers = np.array([(flo + fhi) / 2 for (flo, fhi) in phase_bands], dtype=float)
    amp_centers = np.array([(flo + fhi) / 2 for (flo, fhi) in amp_bands], dtype=float)

    # Compute MI per (phase, amp, epoch)
    mi = np.full((nP, nA, nE), np.nan, dtype=float)
    for e, (a, b) in enumerate(epochs):
        # Slightly shrink window to reduce edge artifacts from Hilbert/filter transients
        pad = int(0.05 * (b - a))  # 5% padding removal
        aa = min(a + pad, b)
        bb = max(a, b - pad)
        if bb - aa < 10:
            continue
        for ip in range(nP):
            ph = phase_filt[ip, aa:bb]
            for ia in range(nA):
                am = amp_filt[ia, aa:bb]
                mi[ip, ia, e] = modulation_index_tort(ph, am, n_bins=n_bins)

    return mi, phase_centers, amp_centers, epoch_times_s


# ------------------------------
# Plotting helpers
# ------------------------------

def plot_comodulogram(
    mi: np.ndarray,
    phase_centers: np.ndarray,
    amp_centers: np.ndarray,
    epoch: Optional[int] = None,
    agg: str = "mean",
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    ax: Optional[plt.Axes] = None,
    cbar: bool = True,
    title: Optional[str] = None,
):
    """Plot a standard comodulogram: phase-freq (x) vs amp-freq (y).

    Parameters
    ----------
    mi : array, shape (nP, nA, nE)
    epoch : int or None
        If None, aggregate across epochs using `agg` ("mean" or "max").
    """
    if mi.ndim != 3:
        raise ValueError("mi must be (nP, nA, nE)")

    if epoch is None:
        if agg == "mean":
            M = np.nanmean(mi, axis=2)
        elif agg == "max":
            M = np.nanmax(mi, axis=2)
        else:
            raise ValueError("agg must be 'mean' or 'max'")
    else:
        if not (0 <= epoch < mi.shape[2]):
            raise IndexError("epoch out of range")
        M = mi[:, :, epoch]

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
    im = ax.imshow(
        M.T,
        origin="lower",
        aspect="auto",
        extent=[phase_centers.min(), phase_centers.max(), amp_centers.min(), amp_centers.max()],
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
    )
    ax.set_xlabel("Phase frequency (Hz)")
    ax.set_ylabel("Amplitude frequency (Hz)")
    if title:
        ax.set_title(title)
    if cbar:
        plt.colorbar(im, ax=ax, label="Modulation Index (MI)")
    return ax


def plot_time_resolved(
    mi: np.ndarray,
    phase_centers: np.ndarray,
    amp_centers: np.ndarray,
    epoch_times_s: Optional[np.ndarray] = None,
    phase_idx: int = 0,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    ax: Optional[plt.Axes] = None,
    cbar: bool = True,
    title: Optional[str] = None,
):
    """Plot time-resolved PAC for a single phase frequency across amplitude bands.

    Shows amplitude-frequency vs time (epochs on x). Useful for stepping through task time.
    """
    if mi.ndim != 3:
        raise ValueError("mi must be (nP, nA, nE)")
    if not (0 <= phase_idx < mi.shape[0]):
        raise IndexError("phase_idx out of range")

    M = mi[phase_idx, :, :]
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4), dpi=120)

    extent = [0, M.shape[1], amp_centers.min(), amp_centers.max()]
    if epoch_times_s is not None and len(epoch_times_s) == M.shape[1]:
        # Use epoch indices but we will relabel ticks with times
        x = np.arange(M.shape[1])
        im = ax.imshow(M, origin="lower", aspect="auto", extent=[x.min(), x.max(), amp_centers.min(), amp_centers.max()], vmin=vmin, vmax=vmax, interpolation="nearest")
        # Relabel x ticks
        ticks = np.linspace(x.min(), x.max(), num=min(8, len(x)))
        ax.set_xticks(ticks)
        ax.set_xticklabels([f"{epoch_times_s[int(t)]:.2f}" for t in ticks])
        ax.set_xlabel("Time (s, epoch centers)")
    else:
        im = ax.imshow(M, origin="lower", aspect="auto", extent=extent, vmin=vmin, vmax=vmax, interpolation="nearest")
        ax.set_xlabel("Epoch index")

    ax.set_ylabel("Amplitude frequency (Hz)")
    if title:
        ax.set_title(title)
    if cbar:
        plt.colorbar(im, ax=ax, label="MI")
    return ax


# ------------------------------
# Minimal example
# ------------------------------

# Synthetic demo (theta phase modulating gamma amplitude)
fs = 1250.0
x = eeg

phase_bands = [(4, 6), (6, 8), (8, 10), (10, 12)]
amp_bands = [(30, 50), (50, 70), (70, 90), (90, 110)]

mi, pf, af, times = compute_pac_comodulogram(x, fs, phase_bands, amp_bands, window_s=1.0, step_s=0.5)

plot_comodulogram(mi, pf, af, agg="mean", title="Comodulogram (mean across epochs)")
plt.show()

plot_time_resolved(mi, pf, af, epoch_times_s=times, phase_idx=2, title=f"Time-resolved MI @ phase~{pf[2]:.1f} Hz")
plt.show()


In [ ]:
"""
Phase–Amplitude Coupling (PAC) comodulogram with Tort Modulation Index (MI)

This module provides:
  1) A robust MI implementation (Tort et al., 2010 style) for limited-time datasets
  2) Time-stepped / event-epoch PAC computation
  3) Publication-ready plotting helpers for comodulograms

Primary entry point: `compute_pac_comodulogram` + `plot_comodulogram`.

Author: ChatGPT
"""
from __future__ import annotations
from typing import Iterable, List, Optional, Sequence, Tuple
import numpy as np
from scipy.signal import butter, filtfilt, hilbert
import matplotlib.pyplot as plt

ArrayLike = np.ndarray

# ------------------------------
# Filtering utilities
# ------------------------------
def _butter_bandpass(low: float, high: float, fs: float, order: int = 4) -> Tuple[np.ndarray, np.ndarray]:
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq
    if low_n <= 0 or high_n >= 1 or low_n >= high_n:
        raise ValueError(f"Invalid band [{low}, {high}] Hz for fs={fs} Hz.")
    b, a = butter(order, [low_n, high_n], btype="bandpass")
    return b, a


def bandpass_filter(x: ArrayLike, fs: float, band: Tuple[float, float], order: int = 4) -> ArrayLike:
    """Zero-phase IIR bandpass using filtfilt.

    Parameters
    ----------
    x : array, shape (n_samples,) or (n_channels, n_samples)
    fs : float
        Sampling rate in Hz.
    band : (low, high)
        Frequency band in Hz.
    order : int
        Butterworth order.
    """
    x = np.asarray(x)
    b, a = _butter_bandpass(band[0], band[1], fs, order)
    if x.ndim == 1:
        return filtfilt(b, a, x)
    elif x.ndim == 2:
        return np.vstack([filtfilt(b, a, xi) for xi in x])
    else:
        raise ValueError("x must be 1D or 2D")


# ------------------------------
# Modulation Index (Tort et al.)
# ------------------------------

def modulation_index_tort(phase_series: ArrayLike, amp_envelope: ArrayLike, n_bins: int = 18, eps: float = 1e-10) -> float:
    """Compute Tort modulation index between a phase time series and an amplitude envelope.

    Parameters
    ----------
    phase_series : array
        Instantaneous phase in radians (e.g., angle(hilbert(theta_band))).
    amp_envelope : array
        Instantaneous amplitude envelope (e.g., abs(hilbert(gamma_band))).
    n_bins : int
        Number of phase bins spanning [-pi, pi).
    eps : float
        Small value to avoid log(0).

    Returns
    -------
    MI : float
        Normalized KL divergence between the phase-binned amplitude distribution and uniform.
    """
    phase = np.asarray(phase_series)
    amp = np.asarray(amp_envelope)
    mask = np.isfinite(phase) & np.isfinite(amp)
    phase = phase[mask]
    amp = amp[mask]
    if phase.size < n_bins * 5:
        return np.nan  # too few samples for a stable estimate

    # Bin by phase
    edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    # Map phase to bin indices [0, n_bins-1]
    idx = np.digitize(phase, edges) - 1
    idx[idx == n_bins] = n_bins - 1

    # Mean amplitude per phase bin
    mean_amp = np.zeros(n_bins)
    counts = np.zeros(n_bins, dtype=int)
    for k in range(n_bins):
        sel = idx == k
        counts[k] = sel.sum()
        if counts[k] > 0:
            mean_amp[k] = amp[sel].mean()
        else:
            mean_amp[k] = 0.0

    if counts.sum() == 0:
        return np.nan

    P = mean_amp / (mean_amp.sum() + eps)
    U = np.full(n_bins, 1.0 / n_bins)

    # KL divergence and normalization
    kl = np.sum(P * (np.log(P + eps) - np.log(U + eps)))
    mi = kl / np.log(n_bins)
    return float(mi)


# ------------------------------
# Epoch helpers
# ------------------------------

def make_sliding_epochs(n_samples: int, fs: float, window_s: float, step_s: Optional[float] = None) -> List[Tuple[int, int]]:
    """Create [start, stop) sample windows covering the signal.

    If step_s is None, defaults to window_s/2.
    """
    if step_s is None:
        step_s = window_s / 2
    w = int(round(window_s * fs))
    s = int(round(step_s * fs))
    if w <= 1:
        raise ValueError("window too small")
    starts = np.arange(0, n_samples - w + 1, s)
    return [(int(a), int(a + w)) for a in starts]


# ------------------------------
# Core PAC computation
# ------------------------------

def compute_pac_comodulogram(
    lfp: ArrayLike,
    fs: float,
    phase_bands: Sequence[Tuple[float, float]],
    amp_bands: Sequence[Tuple[float, float]],
    epochs: Optional[Sequence[Tuple[int, int]]] = None,
    window_s: Optional[float] = None,
    step_s: Optional[float] = None,
    n_bins: int = 18,
    filter_order: int = 4,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Optional[np.ndarray]]:
    """Compute PAC MI comodulogram over frequency pairs and (optionally) time epochs.

    You can provide explicit `epochs` as a list of (start, stop) sample indices.
    Alternatively, set `window_s` (and optional `step_s`) to tile the signal.

    Parameters
    ----------
    lfp : array, shape (n_samples,)
        LFP / field potential time series.
    fs : float
        Sampling rate in Hz.
    phase_bands : list of (f_lo, f_hi)
        Bands that provide phase (typically low frequencies, e.g., theta 4–12 Hz).
    amp_bands : list of (f_lo, f_hi)
        Bands whose amplitude is modulated (typically higher, e.g., gamma 30–200 Hz).
    epochs : list of (start, stop) in samples, optional
        Event-aligned epochs. If None and window_s given, sliding windows are used.
    window_s, step_s : float, optional
        Sliding window parameters in seconds.
    n_bins : int
        Number of phase bins for MI.
    filter_order : int
        Butterworth order for bandpass.

    Returns
    -------
    mi : array, shape (n_phase, n_amp, n_epochs)
    phase_centers : array, shape (n_phase,)
    amp_centers : array, shape (n_amp,)
    epoch_times_s : array, shape (n_epochs,), optional
        Epoch center times in seconds (None if `epochs` provided explicitly without timing context).
    """
    x = np.asarray(lfp).astype(float)
    if x.ndim != 1:
        raise ValueError("lfp must be 1D")

    # Epochs
    if epochs is None:
        if window_s is None:
            raise ValueError("Provide `epochs` or `window_s`.")
        epochs = make_sliding_epochs(len(x), fs, window_s, step_s)
        epoch_times_s = np.array([(a + b) / 2 / fs for a, b in epochs])
    else:
        epoch_times_s = np.array([(a + b) / 2 / fs for a, b in epochs])

    nP = len(phase_bands)
    nA = len(amp_bands)
    nE = len(epochs)

    # Pre-filter once per band to avoid re-filtering per epoch
    phase_filt = []  # list of instantaneous phase arrays
    for (flo, fhi) in phase_bands:
        xf = bandpass_filter(x, fs, (flo, fhi), order=filter_order)
        phase_filt.append(np.angle(hilbert(xf)))
    phase_filt = np.stack(phase_filt, axis=0)  # (nP, n_samples)

    amp_filt = []  # list of amplitude envelopes per amp band
    for (flo, fhi) in amp_bands:
        xf = bandpass_filter(x, fs, (flo, fhi), order=filter_order)
        amp_filt.append(np.abs(hilbert(xf)))
    amp_filt = np.stack(amp_filt, axis=0)  # (nA, n_samples)

    # Centers for axes labels
    phase_centers = np.array([(flo + fhi) / 2 for (flo, fhi) in phase_bands], dtype=float)
    amp_centers = np.array([(flo + fhi) / 2 for (flo, fhi) in amp_bands], dtype=float)

    # Compute MI per (phase, amp, epoch)
    mi = np.full((nP, nA, nE), np.nan, dtype=float)
    for e, (a, b) in enumerate(epochs):
        # Slightly shrink window to reduce edge artifacts from Hilbert/filter transients
        pad = int(0.05 * (b - a))  # 5% padding removal
        aa = min(a + pad, b)
        bb = max(a, b - pad)
        if bb - aa < 10:
            continue
        for ip in range(nP):
            ph = phase_filt[ip, aa:bb]
            for ia in range(nA):
                am = amp_filt[ia, aa:bb]
                mi[ip, ia, e] = modulation_index_tort(ph, am, n_bins=n_bins)

    return mi, phase_centers, amp_centers, epoch_times_s


# ------------------------------
# Plotting helpers
# ------------------------------

def plot_comodulogram(
    mi: np.ndarray,
    phase_centers: np.ndarray,
    amp_centers: np.ndarray,
    epoch: Optional[int] = None,
    agg: str = "mean",
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    ax: Optional[plt.Axes] = None,
    cbar: bool = True,
    title: Optional[str] = None,
):
    """Plot a standard comodulogram: phase-freq (x) vs amp-freq (y).

    Parameters
    ----------
    mi : array, shape (nP, nA, nE)
    epoch : int or None
        If None, aggregate across epochs using `agg` ("mean" or "max").
    """
    if mi.ndim != 3:
        raise ValueError("mi must be (nP, nA, nE)")

    if epoch is None:
        if agg == "mean":
            M = np.nanmean(mi, axis=2)
        elif agg == "max":
            M = np.nanmax(mi, axis=2)
        else:
            raise ValueError("agg must be 'mean' or 'max'")
    else:
        if not (0 <= epoch < mi.shape[2]):
            raise IndexError("epoch out of range")
        M = mi[:, :, epoch]

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
    im = ax.imshow(
        M.T,
        origin="lower",
        aspect="auto",
        extent=[phase_centers.min(), phase_centers.max(), amp_centers.min(), amp_centers.max()],
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
    )
    ax.set_xlabel("Phase frequency (Hz)")
    ax.set_ylabel("Amplitude frequency (Hz)")
    if title:
        ax.set_title(title)
    if cbar:
        plt.colorbar(im, ax=ax, label="Modulation Index (MI)")
    return ax


def plot_time_resolved(
    mi: np.ndarray,
    phase_centers: np.ndarray,
    amp_centers: np.ndarray,
    epoch_times_s: Optional[np.ndarray] = None,
    phase_idx: int = 0,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    ax: Optional[plt.Axes] = None,
    cbar: bool = True,
    title: Optional[str] = None,
):
    """Plot time-resolved PAC for a single phase frequency across amplitude bands.

    Shows amplitude-frequency vs time (epochs on x). Useful for stepping through task time.
    """
    if mi.ndim != 3:
        raise ValueError("mi must be (nP, nA, nE)")
    if not (0 <= phase_idx < mi.shape[0]):
        raise IndexError("phase_idx out of range")

    M = mi[phase_idx, :, :]
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4), dpi=120)

    extent = [0, M.shape[1], amp_centers.min(), amp_centers.max()]
    if epoch_times_s is not None and len(epoch_times_s) == M.shape[1]:
        # Use epoch indices but we will relabel ticks with times
        x = np.arange(M.shape[1])
        im = ax.imshow(M, origin="lower", aspect="auto", extent=[x.min(), x.max(), amp_centers.min(), amp_centers.max()], vmin=vmin, vmax=vmax, interpolation="nearest")
        # Relabel x ticks
        ticks = np.linspace(x.min(), x.max(), num=min(8, len(x)))
        ax.set_xticks(ticks)
        ax.set_xticklabels([f"{epoch_times_s[int(t)]:.2f}" for t in ticks])
        ax.set_xlabel("Time (s, epoch centers)")
    else:
        im = ax.imshow(M, origin="lower", aspect="auto", extent=extent, vmin=vmin, vmax=vmax, interpolation="nearest")
        ax.set_xlabel("Epoch index")

    ax.set_ylabel("Amplitude frequency (Hz)")
    if title:
        ax.set_title(title)
    if cbar:
        plt.colorbar(im, ax=ax, label="MI")
    return ax




# ------------------------------
# Convenience helpers to match the paper-style figure
# ------------------------------

def make_bands(fmin: float, fmax: float, width: float, step: float):
    """Create (low, high) bands from fmin..fmax with given width and step.
    Example: make_bands(2, 30, width=2, step=1) -> 2-Hz wide, 1-Hz stepped.
    """
    starts = np.arange(fmin, fmax - width + 1e-9, step)
    bands = [(float(s), float(s + width)) for s in starts]
    return bands


def plot_comodulogram_paper(
    mi: np.ndarray,
    phase_centers: np.ndarray,
    amp_centers: np.ndarray,
    epoch: Optional[int] = None,
    agg: str = "mean",
    y_dash: Optional[float] = 40.0,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    ax: Optional[plt.Axes] = None,
    title: Optional[str] = None,
    cmap: str = "turbo",
    xlim: Optional[Tuple[float, float]] = (0, 100),
    ylim: Optional[Tuple[float, float]] = (0, 100),
):
    """Paper-style comodulogram (blue background, hot spot, dashed 40 Hz line).

    - phase frequency on x (Hz)
    - amplitude frequency on y (Hz)
    - optional dashed line at ~40 Hz
    - defaults to 0..100 Hz on both axes (override with xlim/ylim)
    """
    if mi.ndim != 3:
        raise ValueError("mi must be (nP, nA, nE)")
    if epoch is None:
        M = np.nanmean(mi, axis=2) if agg == "mean" else np.nanmax(mi, axis=2)
    else:
        M = mi[:, :, epoch]

    if ax is None:
        fig, ax = plt.subplots(figsize=(4.0, 4.0), dpi=150)

    im = ax.imshow(
        M.T,
        origin="lower",
        aspect="auto",
        extent=[phase_centers.min(), phase_centers.max(), amp_centers.min(), amp_centers.max()],
        interpolation="nearest",
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
    )

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    if y_dash is not None:
        ax.axhline(y_dash, linestyle="--", linewidth=1.5, color="white", alpha=0.9)

    ax.set_xlabel("Frequency_phase (Hz)")
    ax.set_ylabel("Frequency_power (Hz)")
    if title:
        ax.set_title(title)

    # Minimal spines/ticks similar to many PAC figures
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.colorbar(im, ax=ax, label="Modulation Index (MI)")
    return ax

fs = 1250.0
x=eeg
phase_bands = make_bands(2, 30, width=2, step=1)
amp_bands   = make_bands(10, 100, width=10, step=2)
mi, pf, af, times = compute_pac_comodulogram(x, fs, phase_bands, amp_bands, window_s=2.0, step_s=1.0)

plot_comodulogram_paper(mi, pf, af, agg="mean", y_dash=40.0, xlim=(0,100), ylim=(0,100), title="PAC comodulogram (paper-style)")
plt.show()


In [ ]:
mwt_zoom